# TrustOCT — Training Notebook 5: Hyperparameter & EDL Optimization Ablations
### Priority Experiments to Beat Baseline Model 2 (Target: > 96.17% Accuracy)
---
Run this notebook on Colab. Executes key Evidential Dirichlet loss hyperparameter experiments:
1. ⭐ **Experiment E (HIGHEST PRIORITY)**: Tuned Low KL Loss Weight (`kl_weight = 0.05`) $\rightarrow$ **Target > 96.5% Accuracy**
2. **Experiment B**: Reduced KL Loss Weight (`kl_weight = 0.3`)
3. **Experiment D**: Extended KL Annealing (`kl_annealing_epochs = 20`)
4. **Experiment A**: Lower Learning Rate (`lr = 5e-5`)


In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    !git pull
    print('Repository updated to latest code.')

try:
    import albumentations
    import ptflops
except ImportError:
    !pip install -r requirements.txt


In [ ]:
import os, sys, time, shutil, zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    print('Running locally or Drive skipped.')


In [ ]:
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload your kaggle.json API token file:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print('Downloading Kermany OCT2017 dataset from Kaggle...')
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print('Dataset downloaded successfully!')
    except Exception as e:
        print(f'Download skipped: {e}')
else:
    print('Kermany dataset already present on disk!')


In [ ]:
from src.datasets.dataset import auto_detect_columns, patient_level_split

csv_path = 'kermany_dataset_mapping.csv'
if not os.path.exists(csv_path):
    root_oct = '/content/Kermany/OCT2017 '
    if not os.path.exists(root_oct): root_oct = '/content/Kermany'
    if not os.path.exists(root_oct): root_oct = '/content/OCT2017'
    if os.path.exists(root_oct):
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files_list in os.walk(root_oct):
            for f in files_list:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({'image_path': os.path.join(root, f), 'label': lbl, 'patient_id': pt_id})
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        df_new.to_csv(csv_path, index=False)

df = auto_detect_columns(pd.read_csv(csv_path))
is_colab = os.path.exists('/content')
local_kermany = '/content/Kermany'
local_oct2017 = '/content/OCT2017'
if is_colab and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
    def force_local_path(path_str):
        p = path_str.replace('\\', '/').replace('//', '/')
        parts = p.split('/')
        for folder in ['train', 'val', 'test']:
            if folder in parts:
                idx = parts.index(folder)
                subpath = '/'.join(parts[idx:])
                for candidate in [
                    os.path.join(local_kermany, subpath),
                    os.path.join(local_kermany, 'OCT2017', subpath),
                    os.path.join(local_kermany, 'OCT2017 ', subpath),
                    os.path.join(local_oct2017, subpath)
                ]:
                    if os.path.exists(candidate): return candidate
        return path_str
    df['image_path'] = df['image_path'].apply(force_local_path)

train_df, val_df, test_df = patient_level_split(df)
print(f'Train cohort: {len(train_df)}, Val cohort: {len(val_df)}, Test cohort: {len(test_df)}')


### ⭐ Step 0 (TOP PRIORITY): Experiment E – Optimal Low KL Weight (`kl_weight = 0.05`)
**Goal**: Reduce KL penalty to allow full evidence magnitude expansion, driving test accuracy past 96.17%.


In [ ]:
from src.training.trainer import run_experiment

print('🚀 Executing Experiment E (kl_weight = 0.05)...')
run_experiment(
    'trustoct_expE',
    train_df, val_df, test_df,
    epochs=30, lr=1e-4, batch_size=32,
    patience=5, kl_weight=0.05, kl_annealing_epochs=10,
    device_name=str(device)
)


### ⭐ Step 1 (Highest Priority): Experiment B – Reduced KL Loss Weight (`kl_weight = 0.3`)
Tests whether reducing the KL divergence penalty prevents uncertainty regularization from overpowering classification accuracy.

In [ ]:
from src.training.trainer import run_experiment
print('🚀 Executing Experiment B (kl_weight = 0.3)...')
run_experiment(
    experiment_id='trustoct_expB',
    train_df=train_df, val_df=val_df, test_df=test_df,
    epochs=30, lr=1e-4, batch_size=32, device_name=str(device),
    patience=5, kl_weight=0.3, kl_annealing_epochs=10
)


### ⭐ Step 2 (High Priority): Experiment D – Extended KL Annealing (`kl_annealing_epochs = 20`)
Slows down KL regularization to allow the backbone to form strong discriminative features before enforcing Dirichlet uncertainty.

In [ ]:
print('🚀 Executing Experiment D (kl_annealing_epochs = 20)...')
run_experiment(
    experiment_id='trustoct_expD',
    train_df=train_df, val_df=val_df, test_df=test_df,
    epochs=30, lr=1e-4, batch_size=32, device_name=str(device),
    patience=5, kl_weight=1.0, kl_annealing_epochs=20
)


### Step 3: Experiment A – Lower Learning Rate (`lr = 5e-5`)
Tests whether smoother optimization improves Dirichlet evidence learning.

In [ ]:
print('🚀 Executing Experiment A (lr = 5e-5)...')
run_experiment(
    experiment_id='trustoct_expA',
    train_df=train_df, val_df=val_df, test_df=test_df,
    epochs=30, lr=5e-5, batch_size=32, device_name=str(device),
    patience=5, kl_weight=1.0, kl_annealing_epochs=10
)


### Step 4: Experiment C – Increased Early Stopping Patience (`patience = 8`)
Tests whether giving EDL models more epochs allows convergence to a better optimum.

In [ ]:
print('🚀 Executing Experiment C (patience = 8)...')
run_experiment(
    experiment_id='trustoct_expC',
    train_df=train_df, val_df=val_df, test_df=test_df,
    epochs=30, lr=1e-4, batch_size=32, device_name=str(device),
    patience=8, kl_weight=1.0, kl_annealing_epochs=10
)


## 📊 Publication Decision Matrix (Internal + Calibration + OCTID External + Stability)
Evaluates Baseline vs Experiments B, D, A, and C across **Internal Accuracy**, **Macro F1**, **Kappa**, **ECE**, **Brier**, **OCTID External Accuracy**, **OCTID F1**, **Best Epoch**, and **Best Val Loss**.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef, cohen_kappa_score
from src.evaluation.calibration import calculate_ece, calculate_brier_score
from src.models.builder import build_model
from src.evaluation.cross_dataset import run_external_validation

CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}
def parse_col(series): return series.apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0))).values

# 1. Load OCTID external cohort if available
octid_root = None
for c in ['/content/OCTID', '/content/drive/MyDrive/OCTID', '/content/drive/MyDrive/TrustOCT_Results/OCTID']:
    if os.path.exists(c):
        octid_root = c
        break

df_octid = pd.DataFrame()
if octid_root:
    records_octid = []
    class_map_octid = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'amd': 2}
    for root, dirs, files in os.walk(octid_root):
        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                parent_dir = os.path.basename(root).lower()
                lbl = -1
                for k, v in class_map_octid.items():
                    if k in parent_dir: lbl = v; break
                if lbl != -1: records_octid.append({'image_path': os.path.join(root, f), 'label': lbl})
    df_octid = pd.DataFrame(records_octid)

ablation_runs = [
    ('outputs/trustoct', 'TrustOCT Baseline'),
    ('outputs/trustoct_expB', 'Exp B (kl_weight = 0.3)'),
    ('outputs/trustoct_expD', 'Exp D (annealing = 20)'),
    ('outputs/trustoct_expA', 'Exp A (lr = 5e-5)'),
    ('outputs/trustoct_expC', 'Exp C (patience = 8)')
]

rows = []
for out_dir, display_name in ablation_runs:
    pred_path = os.path.join(out_dir, 'predictions.csv')
    metrics_path = os.path.join(out_dir, 'metrics.csv')
    weights_path = os.path.join(out_dir, 'weights_best.pth')
    
    if os.path.exists(pred_path):
        df_p = pd.read_csv(pred_path)
        y_true = parse_col(df_p['true_label'])
        y_pred = parse_col(df_p['pred_label'])
        probs = df_p[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
        u_arr = df_p['uncertainty'].values if 'uncertainty' in df_p.columns else None
        
        acc = accuracy_score(y_true, y_pred)
        _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        kappa = cohen_kappa_score(y_true, y_pred)
        
        conf = (1.0 - u_arr) * np.max(probs, axis=1) if u_arr is not None else np.max(probs, axis=1)
        ece = calculate_ece(conf, (y_pred == y_true).astype(int))
        brier = calculate_brier_score(probs, y_true)
        
        best_epoch, best_val_loss = 'N/A', 'N/A'
        if os.path.exists(metrics_path):
            try:
                df_m = pd.read_csv(metrics_path)
                if 'val_loss' in df_m.columns:
                    min_row = df_m.loc[df_m['val_loss'].idxmin()]
                    best_epoch = int(min_row.get('epoch', 0))
                    best_val_loss = f"{min_row['val_loss']:.4f}"
            except Exception:
                pass
                
        # OCTID External Evaluation
        octid_acc_str, octid_f1_str = 'N/A', 'N/A'
        if os.path.exists(weights_path) and len(df_octid) > 0:
            try:
                cfg = {'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}
                m_inst = build_model(cfg).to(device)
                m_inst.load_state_dict(torch.load(weights_path, map_location=device))
                m_inst.eval()
                res_ext = run_external_validation(m_inst, df_octid, batch_size=32, is_evidential=True, device_name=str(device))
                y_ext_t = np.array(res_ext['labels'])
                y_ext_p = np.array(res_ext['predictions'])
                present_c = sorted(list(np.unique(y_ext_t)))
                ext_acc = accuracy_score(y_ext_t, y_ext_p)
                _, _, ext_f1, _ = precision_recall_fscore_support(y_ext_t, y_ext_p, labels=present_c, average='macro', zero_division=0)
                octid_acc_str = f"{ext_acc*100:.2f}%"
                octid_f1_str = f"{ext_f1:.4f}"
            except Exception as e:
                print(f'OCTID evaluation error for {display_name}: {e}')
                
        rows.append({
            'Experiment': display_name,
            'Best Epoch': best_epoch,
            'Best Val Loss': best_val_loss,
            'Accuracy (%)': f"{acc*100:.2f}%",
            'Macro F1': f"{f1:.4f}",
            'Kappa': f"{kappa:.4f}",
            'ECE ↓': f"{ece:.4f}",
            'Brier ↓': f"{brier:.4f}",
            'OCTID Acc (%)': octid_acc_str,
            'OCTID F1': octid_f1_str
        })

if len(rows) > 0:
    matrix_df = pd.DataFrame(rows)
    print('\n--- TRUSTOCT PUBLICATION-GRADE DECISION MATRIX ---')
    display(matrix_df)
    os.makedirs('results/tables', exist_ok=True)
    matrix_df.to_csv('results/tables/trustoct_ablation_decision_matrix.csv', index=False)

# Sync to Google Drive
shared_folder = '/content/drive/MyDrive/TrustOCT_Results/outputs'
os.makedirs(shared_folder, exist_ok=True)
for exp_id in ['trustoct_expB', 'trustoct_expD', 'trustoct_expA', 'trustoct_expC']:
    src = f'outputs/{exp_id}'
    dst = os.path.join(shared_folder, exp_id)
    if os.path.exists(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Synced {exp_id} to shared Google Drive.')
